In [1]:
import requests
from bs4 import BeautifulSoup

In [2]:
# 対象のURL
url = 'https://books.toscrape.com/'

# WebページにアクセスしてHTMLを取得
response = requests.get(url)

# BeautifulSoupでHTMLをパース（解析）
soup = BeautifulSoup(response.text, 'html.parser')

# ページタイトルを確認
print(soup.title.text)


    All products | Books to Scrape - Sandbox



In [3]:
soup

<!DOCTYPE html>

<!--[if lt IE 7]>      <html lang="en-us" class="no-js lt-ie9 lt-ie8 lt-ie7"> <![endif]-->
<!--[if IE 7]>         <html lang="en-us" class="no-js lt-ie9 lt-ie8"> <![endif]-->
<!--[if IE 8]>         <html lang="en-us" class="no-js lt-ie9"> <![endif]-->
<!--[if gt IE 8]><!--> <html class="no-js" lang="en-us"> <!--<![endif]-->
<head>
<title>
    All products | Books to Scrape - Sandbox
</title>
<meta content="text/html; charset=utf-8" http-equiv="content-type"/>
<meta content="24th Jun 2016 09:29" name="created"/>
<meta content="" name="description"/>
<meta content="width=device-width" name="viewport"/>
<meta content="NOARCHIVE,NOCACHE" name="robots"/>
<!-- Le HTML5 shim, for IE6-8 support of HTML elements -->
<!--[if lt IE 9]>
        <script src="//html5shim.googlecode.com/svn/trunk/html5.js"></script>
        <![endif]-->
<link href="static/oscar/favicon.ico" rel="shortcut icon"/>
<link href="static/oscar/css/styles.css" rel="stylesheet" type="text/css"/>
<link href="s

In [4]:
# カテゴリの親リスト（nav-list）の中を探す
category_section = soup.find('ul', class_='nav nav-list')

# ul > li > ul > li > aの形でカテゴリが並んでる
category_links = category_section.find_all('a')

# カテゴリ名とURLの一覧を作成
categories = []

for link in category_links:
    name = link.text.strip()
    href = link.get('href')

    #トップカテゴリ（”Books"）はのぞく
    if name.lower() != "books":
        full_url = url + href
        categories.append({
            'name': name,
            'url': full_url
        })

#結果を確認
for cat in categories[:5]:
    print(f"{cat['name']} → {cat['url']}")

Travel → https://books.toscrape.com/catalogue/category/books/travel_2/index.html
Mystery → https://books.toscrape.com/catalogue/category/books/mystery_3/index.html
Historical Fiction → https://books.toscrape.com/catalogue/category/books/historical-fiction_4/index.html
Sequential Art → https://books.toscrape.com/catalogue/category/books/sequential-art_5/index.html
Classics → https://books.toscrape.com/catalogue/category/books/classics_6/index.html


In [6]:
# 最初のカテゴリ（Travelなど）のページへアクセスし、HTMLを取得
test_category = categories[0]
cat_url = test_category['url']

# カテゴリページにアクセス
cat_response = requests.get(cat_url)
cat_soup = BeautifulSoup(cat_response.text, 'html.parser')

# タイトルで確認
print("カテゴリ名：", test_category['name'])
print("ページタイトル：", cat_soup.title.text)

カテゴリ名： Travel
ページタイトル： 
    Travel | 
     Books to Scrape - Sandbox




In [7]:
# カテゴリページから商品一覧を取得
books = cat_soup.find_all('article', class_='product_pod')

# 商品データを格納するリスト
book_list = []

for book in books:
    #タイトル
    title = book.h3.a['title']
    #価格
    price = book.find('p', class_='price_color').text
    #在庫状況
    availability = book.find('p', class_='instock availability').text.strip()

    #相対URLを取得　→　絶対URLに変更
    rel_url = book.h3.a['href']
    detail_url = 'https://books.toscrape.com/catalogue/' + rel_url.replace("../", "")

    #辞書形式で格納
    book_list.append({
        "タイトル": title,
        "価格": price,
        "在庫状況": availability,
        "詳細URL": detail_url
    })

# 上位3件だけ確認
for book in book_list[:3]:
    print(book)

{'タイトル': "It's Only the Himalayas", '価格': 'Â£45.17', '在庫状況': 'In stock', '詳細URL': 'https://books.toscrape.com/catalogue/its-only-the-himalayas_981/index.html'}
{'タイトル': 'Full Moon over Noahâ\x80\x99s Ark: An Odyssey to Mount Ararat and Beyond', '価格': 'Â£49.43', '在庫状況': 'In stock', '詳細URL': 'https://books.toscrape.com/catalogue/full-moon-over-noahs-ark-an-odyssey-to-mount-ararat-and-beyond_811/index.html'}
{'タイトル': 'See America: A Celebration of Our National Parks & Treasured Sites', '価格': 'Â£48.87', '在庫状況': 'In stock', '詳細URL': 'https://books.toscrape.com/catalogue/see-america-a-celebration-of-our-national-parks-treasured-sites_732/index.html'}


In [15]:
from urllib.parse import urljoin

# 全ての書籍情報を格納するリスト
all_books = []

# 初期ページURL
current_page = test_category['url']

while True:
    #ページ取得
    res = requests.get(current_page)
    soup = BeautifulSoup(res.text, 'html.parser')

    # 書籍取得
    books = soup.find_all('article', class_='product_pod')

    for book in books:
        title = book.h3.a['title']
        price = book.find('p', class_='price_color').text
        availability = book.find('p', class_='instock availability').text.strip()
        rel_url = book.h3.a['href']
        detail_url = 'https://books.toscrape.com/catalogue/' + rel_url.replace("../", "")

        all_books.append({
            "タイトル": title,
            "価格": price,
            "在庫状況": availability,
            "詳細URL": detail_url
        })

    # 次ページがあるか確認
    next_btn = soup.find('li', class_='next')
    if next_btn:
        next_link = next_btn.a['href']
        # urljoinを使って絶対URLに変換
        current_page = urljoin(current_page, next_link)
    else:
        break

#書籍数を確認
print(f"{test_category['name']}カテゴリで手足した書籍数：{len(all_books)}")
print("最初の3件：")
for b in all_books[:3]:
    print(b)

Travelカテゴリで手足した書籍数：11
最初の3件：
{'タイトル': "It's Only the Himalayas", '価格': 'Â£45.17', '在庫状況': 'In stock', '詳細URL': 'https://books.toscrape.com/catalogue/its-only-the-himalayas_981/index.html'}
{'タイトル': 'Full Moon over Noahâ\x80\x99s Ark: An Odyssey to Mount Ararat and Beyond', '価格': 'Â£49.43', '在庫状況': 'In stock', '詳細URL': 'https://books.toscrape.com/catalogue/full-moon-over-noahs-ark-an-odyssey-to-mount-ararat-and-beyond_811/index.html'}
{'タイトル': 'See America: A Celebration of Our National Parks & Treasured Sites', '価格': 'Â£48.87', '在庫状況': 'In stock', '詳細URL': 'https://books.toscrape.com/catalogue/see-america-a-celebration-of-our-national-parks-treasured-sites_732/index.html'}


In [19]:
import csv
import os

# 出力フォルダ作成(無ければ)
os.makedirs('book_csv', exist_ok=True)

#　　全てのカテゴリをループ
for category in categories:
    print(f"カテゴリ処理中：{category['name']}")

    current_page = category['url']
    all_books = []

    while True:
        res = requests.get(current_page)
        soup = BeautifulSoup(res.text, 'html.parser')

        books = soup.find_all('article', class_='product_pod')

        for book in books:
            title = book.h3.a['title']
            price = book.find('p', class_='price_color').text
            availability = book.find('p', class_='instock availability')
            rel_url = book.h3.a['href']
            detail_url = 'https://books.toscrape.com/catalogue/' + rel_url.replace('../', '')

            all_books.append({
                'タイトル': title,
                '価格': price,
                '在庫状況': availability,
                '詳細URL': detail_url
            })

        next_btn = soup.find('li', class_='next')
        if next_btn:
            next_link = next_btn.a['href']
            current_page = urljoin(current_page, next_link)
        else:
            break

    #カテゴリ名でCSVファイルを保存
    filename = f"book_csv/{category['name'].replace(' ', '_')}.csv"
    with open(filename, mode='w', encoding='utf-8', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=['タイトル', '価格', '在庫状況', '詳細URL'])
        writer.writeheader()
        writer.writerows(all_books)

    print(f"→ 保存完了:{filename}({len(all_books)}冊)")

カテゴリ処理中：Travel
→ 保存完了:book_csv/Travel.csv(11冊)
カテゴリ処理中：Mystery
→ 保存完了:book_csv/Mystery.csv(32冊)
カテゴリ処理中：Historical Fiction
→ 保存完了:book_csv/Historical_Fiction.csv(26冊)
カテゴリ処理中：Sequential Art
→ 保存完了:book_csv/Sequential_Art.csv(75冊)
カテゴリ処理中：Classics
→ 保存完了:book_csv/Classics.csv(19冊)
カテゴリ処理中：Philosophy
→ 保存完了:book_csv/Philosophy.csv(11冊)
カテゴリ処理中：Romance
→ 保存完了:book_csv/Romance.csv(35冊)
カテゴリ処理中：Womens Fiction
→ 保存完了:book_csv/Womens_Fiction.csv(17冊)
カテゴリ処理中：Fiction
→ 保存完了:book_csv/Fiction.csv(65冊)
カテゴリ処理中：Childrens
→ 保存完了:book_csv/Childrens.csv(29冊)
カテゴリ処理中：Religion
→ 保存完了:book_csv/Religion.csv(7冊)
カテゴリ処理中：Nonfiction
→ 保存完了:book_csv/Nonfiction.csv(110冊)
カテゴリ処理中：Music
→ 保存完了:book_csv/Music.csv(13冊)
カテゴリ処理中：Default
→ 保存完了:book_csv/Default.csv(152冊)
カテゴリ処理中：Science Fiction
→ 保存完了:book_csv/Science_Fiction.csv(16冊)
カテゴリ処理中：Sports and Games
→ 保存完了:book_csv/Sports_and_Games.csv(5冊)
カテゴリ処理中：Add a comment
→ 保存完了:book_csv/Add_a_comment.csv(67冊)
カテゴリ処理中：Fantasy
→ 保存完了:book_csv/Fantasy.csv(48冊)
カテゴリ処理中：